In [ ]:
#| hide
from yttoc.core import *

# yttoc

> YouTube xscript to structured Table of Contents — turn any YouTube video into searchable, timestamped knowledge

`yttoc` downloads YouTube captions (Japanese preferred, English fallback), parses them into normalized transcript segments, and uses an LLM to generate a structured Table of Contents with section summaries, keywords, and evidence quotes.

**What you get from a single video URL:**

- Timestamped transcript (`yttoc-raw`)
- Auto-generated Table of Contents with deep links (`yttoc-toc`)
- Per-section summaries with keywords and quoted evidence (`yttoc-sum`)

## Usage

### Installation

```sh
pip install git+https://github.com/doyu/yttoc.git
```

**Requirements:**
- Python 3.9+
- [yt-dlp](https://github.com/yt-dlp/yt-dlp) (installed automatically)
- `OPENAI_API_KEY` environment variable set (for `yttoc-toc` and `yttoc-sum`)

## Example: Andrej Karpathy on Code Agents (66 min)

Fetch the video and generate a full ToC + summaries in three commands:

```bash
vid=$(yttoc-fetch https://www.youtube.com/watch?v=kwSVtQ7dziU)
yttoc-toc "$vid"
yttoc-sum "$vid"
```

### `yttoc-toc` — Table of Contents

```
# Skill Issue: Andrej Karpathy on Code Agents, AutoResearch, and the Loopy Era of AI
Channel: No Priors: AI, Machine Learning, Tech, & Startups | Duration: 1:06:31 | 20260320

1. AI agents change coding overnight 0:00-3:24 (3:24) https://youtube.com/watch?v=kwSVtQ7dziU&t=0
2. Parallel agents, token throughput, and the new bottleneck 3:24-6:49 (3:25) ...&t=204
3. From agents to persistent 'claws' 6:49-9:25 (2:36) ...&t=409
...
6. AutoResearch and removing humans from the loop 16:33-21:00 (4:27) ...&t=993
...
14. MicroGPT and teaching agents instead of humans 1:01:31-1:06:31 (5:00) ...&t=3691
```

Each line links directly to that section in the YouTube player.

### `yttoc-sum --section 6` — Section Summary

```
## 6. AutoResearch and removing humans from the loop (16:33)
AutoResearch is Karpathy's attempt to remove humans from the research loop
by letting agents pursue objectives autonomously for long periods. He found
that even on a well-tuned small LLM training setup, the system discovered
improvements he had missed, reinforcing his belief in recursive self-improvement.
**Keywords:** AutoResearch, recursive self-improvement, hyperparameter tuning,
LLM training, DataChat, validation loss, objective metrics, automation
**Evidence:** "I don't want to be like the researcher in the loop...
I'm holding the system back" [17:17]
```

### `yttoc-raw --section 6` — Section Transcript

```
## 6. AutoResearch and removing humans from the loop (16:33 - 21:00)
[16:33] Auto research. Yeah. So I think like I
[16:36] had a tweet earlier where I kind of like
[16:38] said something along the lines of to get
[16:40] the most out of the tools that have
[16:41] become available now you have to remove
[16:43] yourself as the as the bottleneck.
...
```

### All commands

```bash
yttoc-fetch <url>              # Download captions (ja preferred, en fallback)
yttoc-list                     # Show cached videos
yttoc-raw <video_id>           # Full transcript
yttoc-raw <vid> --section 3    # Section transcript
yttoc-toc <video_id>           # Generate Table of Contents
yttoc-sum <video_id>           # All section summaries + full summary
yttoc-sum <vid> --section 3    # Single section summary
yttoc-map <id1> <id2> ...      # Course-wide learning map across multiple videos
```

## Course maps across multiple videos

Once several videos in a course are cached, `yttoc-map` builds a single Markdown learning map with three views — **By Lecture** / **By Topic** / **By Keyword** — each section linking to the YouTube player at the right timestamp.

```bash
yttoc-map $(cat course-ids.txt) --title "My Course Map" > course-map.md
```

Render it as an interactive folding mindmap (single offline HTML file):

```bash
npx -y markmap-cli@latest --offline course-map.md -o course-map.html
xdg-open course-map.html
```

…or as a static linked page:

```bash
pandoc course-map.md -s --toc -o course-map.html
```

### Searching with fzf

`yttoc-map` is for **browsing**. For **finding** a specific topic across the whole course, pipe `summaries.json` through `jq` into `fzf` — fuzzy-search section titles in the left pane, see summary + keywords in the preview pane below, hit Enter to open the deep-linked YouTube URL.

```bash
yttoc-find() {
    for id in "$@"; do
        jq -r --arg id "$id" '
            .video.url as $u
            | .sections[]
            | [
                ($u + "&t=" + (.start|tostring)),
                ($id + " §" + .path + " " + .title),
                .summary,
                (.keywords | join(", "))
            ] | @tsv
        ' "$HOME/.cache/yttoc/$id/summaries.json"
    done \
    | fzf --delimiter=$'\t' \
          --with-nth=2 \
          --preview 'printf "%s\n\nKeywords: %s\n" {3} {4}' \
          --preview-window=down:60%:wrap \
          --bind 'enter:execute(xdg-open {1})'
}
```

Usage — same shell-expansion style as `yttoc-map`:

```bash
yttoc-find $(cat course-ids.txt)
yttoc-find $(ls ~/.cache/yttoc)
```

Drop into `~/.bashrc`. Replace `xdg-open` with `open` on macOS.

The fzf snippet stays out of the Python package on purpose — `xdg-open`/`open` is environment-specific, fzf isn't a Python dep, and shell composition is what shells are good at. yttoc generates the data; your shell does the searching.

## Development

Built with [nbdev](https://nbdev.fast.ai/). Source of truth is in `nbs/` notebooks.

```sh
pip install -e .
nbdev_prepare    # export, test, build docs
```